# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# NOTE: dataset.metadata is an object; access attributes directly
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")
# Optionally show additional metadata
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We begin by inspecting what record sets and fields are present. All references use the `@id` for each entity.

In [ ]:
# List all record sets and their @ids
record_sets = dataset.metadata.recordSet
if record_sets:
    print("Record sets:")
    for rs in record_sets:
        print(f"  @id: {rs['@id']}, name: {rs.get('name', 'No name')}")

    # List fields for each record set
    for rs in record_sets:
        print(f"\nFields in record set {rs['@id']}:")
        fields = rs.get('field', [])
        for field in fields:
            print(f"  Field @id: {field['@id']} - name: {field.get('name', field['@id'])} - dataType: {field.get('dataType', '')}")
else:
    print("No record sets found in the metadata. The dataset schema might have fields elsewhere or in the distributions.")
    print("Try listing distributions:")
    distributions = dataset.metadata.distribution
    for d in distributions:
        print(f"Distribution @id: {d['@id']}")
        # If columns info is available, print columns
        if 'column' in d:
            print(f"Columns:")
            for col in d['column']:
                print(f"  Column @id: {col['@id']} - name: {col.get('name', col['@id'])} - dataType: {col.get('dataType', '')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

All code references entities by their `@id` as required.

In [ ]:
# Depending on the metadata, define record_set @ids
# For this dataset, assume tabular data stored in distribution(s), use those @id(s)
record_set_ids = []
if dataset.metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
else:
    # Fallback: use distribution @id
    record_set_ids = [d['@id'] for d in dataset.metadata.distribution]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records and isinstance(records[0], dict):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set @id: {record_set_id}")
            print(f"Data columns: {df.columns.tolist()}")
            print(f"Sample data:\n{df.head()}\n")
        else:
            print(f"No record data from {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Choose one record set id for further exploration
# If multiple, pick the first
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
else:
    main_record_set_id = None
print(f"Selected record set @id for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

Here, we demonstrate filtering, normalizing, and grouping using the columns available in the selected DataFrame.

All fields and columns are referenced by their `@id` where available.

In [ ]:
# Choose a numeric field for filtering
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to detect a numeric column by dtype
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use the first numeric column
        print(f"Numeric field selected (by column name): {numeric_field}")
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normalized_col]].head())

        # Pick a group-able field (categorical)
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}, mean of {numeric_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No main record set loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib to plot histograms and barplots from the EDA section.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_cols:
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If grouping, barplot
    if group_fields:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to ingest and explore the FAIR^2 colorectal cancer survivor dataset, referencing all entities by their `@id` as required. Data loading, overview, filtering, normalization, grouping, and visualization steps are provided. You can extend this notebook for deeper statistical analysis or modeling tasks.